# Portfolio Metrics Extraction — Demo

Runs the pipeline over the sample PDF folder and reviews the outputs.
See [specs/spec.md](specs/spec.md) for design rationale and the README for the pipeline diagram.

In [1]:
# 1. Run the pipeline (requires OPENAI_API_KEY in .env)
PDF_FOLDER = "/path/to/pdf-folder"  # <- set me
# !python -m src.pipeline --input {PDF_FOLDER}

In [2]:
# 2. The fact table — every row traceable to file + page + verbatim label
import pandas as pd
metrics = pd.read_csv("output/metrics_long.csv")
metrics.head(15)

,company,company_as_reported,canonical_metric,verbatim_label,value,unit,currency,quarter,year,period_basis,source_file,source_type,report_period,page,location,note,non_canonical,superseded
0,ApexFreight,Apex Freight Solutions Inc.,NaN,Completed Shipments,1.28M,M,NaN,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,True,False
1,ApexFreight,Apex Freight Solutions Inc.,NaN,Active Shippers,"3,460",count,NaN,2.0,2025.0,point_in_time,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,True,False
2,ApexFreight,Apex Freight Solutions Inc.,NaN,Active Carriers,"7,900",count,NaN,2.0,2025.0,point_in_time,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,True,False
3,ApexFreight,Apex Freight Solutions Inc.,NaN,On-Time Delivery Rate,96.1%,%,NaN,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,Carrier-reported with a 48-hour grace window p...,True,False
4,ApexFreight,Apex Freight Solutions Inc.,NaN,Average Take Rate,11.2%,%,NaN,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,True,False
5,ApexFreight,Apex Freight Solutions Inc.,revenue,Recognized Revenue (transaction),8.6M,M,USD,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,Disclosed separately for the first time in Q2 ...,False,False
6,ApexFreight,Apex Freight Solutions Inc.,NaN,SaaS Tool Fee Revenue,0.7M,M,USD,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,Disclosed separately for the first time in Q2 ...,True,False
7,ApexFreight,Apex Freight Solutions Inc.,revenue,Total Recognized Revenue,9.3M,M,USD,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,False,False
8,ApexFreight,Apex Freight Solutions Inc.,gross_margin,Gross Margin,54%,%,NaN,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,Expanded to 54%.,False,False
9,ApexFreight,Apex Freight Solutions Inc.,NaN,Marketing Spend as % of Revenue,16%,%,NaN,2.0,2025.0,quarterly,ApexFreight_Q2_2025.pdf,standalone,Q2 2025,1,table,NaN,True,False


In [3]:
# 3. Companies x quarters — the human-review view
pd.read_csv("output/pivot.csv")

,company,canonical_metric,Q2 2024,Q3 2024,Q4 2024,Q1 2025,Q2 2025
0,ApexFreight,gross_margin,NaN,NaN,51%,52%,54%
1,ApexFreight,headcount,NaN,NaN,196,199,204
2,ApexFreight,revenue,NaN,NaN,9.0M,8.9M,8.6M | 9.3M
3,CarbonTrack,arr,NaN,NaN,NaN,$15.2M,$16.9M
4,CarbonTrack,gross_margin,NaN,NaN,NaN,72%,73%
5,CarbonTrack,headcount,NaN,NaN,NaN,72,78
6,CarbonTrack,net_revenue_retention,NaN,NaN,NaN,118%,121%
7,CarbonTrack,revenue,NaN,NaN,NaN,$3.8M,$4.1M
8,ClearPay,gross_margin,NaN,NaN,NaN,66%,67%
9,ClearPay,headcount,NaN,NaN,NaN,203,211


In [4]:
# 4. What the pipeline is NOT sure about
pd.read_csv("output/flags.csv").sort_values("severity")

,severity,source_file,company,metric,period,detail
0,info,CarbonTrack_Q2_2025.pdf,CarbonTrack,arr,Q2 2025,reported in 2 sources ['CarbonTrack_Q2_2025.pd...
25,info,NovaCloud_Q2_2025.pdf,NovaCloud,revenue,Q2 2025,reported in 2 sources ['NovaCloud_Q2_2025.pdf'...
26,info,NovaCloud_Q2_2025.pdf,NovaCloud,arr,Q2 2025,snapshot and standalone report agree
27,info,NovaCloud_Q2_2025.pdf,NovaCloud,cash,Q2 2025,snapshot and standalone report agree
29,info,NovaCloud_Q2_2025.pdf,NovaCloud,headcount,Q2 2025,snapshot and standalone report agree
30,info,NovaCloud_Q2_2025.pdf,NovaCloud,logo_churn,Q2 2025,snapshot and standalone report agree
31,info,NovaCloud_Q2_2025.pdf,NovaCloud,net_revenue_retention,Q2 2025,snapshot and standalone report agree
32,info,NovaCloud_Q2_2025.pdf,NovaCloud,revenue,Q2 2025,snapshot and standalone report agree
33,info,NovaCloud_Q3_2024.pdf,NovaCloud,revenue,Q3 2024,same value restated under multiple labels/loca...
34,info,PeopleFlow_Q1_2025.pdf,PeopleFlow,gross_margin,Q1 2025,reported in 2 sources ['PeopleFlow_Q1_2025.pdf...


## Trace one number to its source

Pick any value in the pivot, find its row in `metrics_long.csv`, and open the source PDF
at the recorded page — the verbatim label is printed there. The verify step does this
string-match automatically for every extracted value.